# Outbound Domain Traffic

This notebook explores outbound click traffic from `lifeline.org.au` pages to other domains using the GA4 BigQuery export.

It can run sitewide, for a single source page, or for a wildcard page-path set such as `/get-help/support-toolkit*`.

### Quick start
1. Run the **Setup (run once)** cell.
2. Adjust the **Parameters** if you want a narrower page-path scope or to include Lifeline subdomains.
3. Run the query cell and confirm the returned domains look sensible.
4. Review the summary table, daily trend chart, and total-clicks chart.

**Metric:** outbound click events  
**Data source:** GA4 BigQuery export (`events_*`)  
**Important caveat:** this analysis relies on outbound clicks being present in GA4 as `click` events with `link_domain` or `link_url` populated.


In [ ]:
#@title Setup (run once)
import os
import re
import sys
from datetime import date, timedelta

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidoanto/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q 'plotly>=6.1.1' 'kaleido>=1.2.0'
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
import plotly.express as px
from google.cloud import bigquery

import lifeline_theme
from lla_data import config
from lla_data.bq import dry_run_bytes, get_client, load_sql_template, run_query

lifeline_theme.inject_fonts()

client = get_client()


In [ ]:
#@title Parameters
ANALYSIS_START_DATE = date(2025, 12, 15) #@param {type:"date"}
ANALYSIS_END_DATE = date.today() - timedelta(days=1) #@param {type:"date"}
PAGE_PATH_FILTER = "" #@param {type:"string"}
INCLUDE_LIFELINE_SUBDOMAINS = False #@param {type:"boolean"}
TOP_N_DOMAINS = 10 #@param {type:"integer"}

if TOP_N_DOMAINS < 1:
    raise ValueError("TOP_N_DOMAINS must be at least 1.")
if ANALYSIS_END_DATE < ANALYSIS_START_DATE:
    raise ValueError("ANALYSIS_END_DATE must be on or after ANALYSIS_START_DATE.")

def normalize_page_path_filter(value: str) -> str:
    value = (value or "").strip().lower()
    if not value:
        return ""
    if not value.startswith("/"):
        value = f"/{value}"
    if value != "/":
        value = re.sub(r"/$", "", value)
    return value

PAGE_PATH_FILTER = normalize_page_path_filter(PAGE_PATH_FILTER)
analysis_window_label = f"{ANALYSIS_START_DATE:%Y-%m-%d} to {ANALYSIS_END_DATE:%Y-%m-%d}"
subdomain_scope_label = "including Lifeline subdomains" if INCLUDE_LIFELINE_SUBDOMAINS else "excluding Lifeline subdomains"

if not PAGE_PATH_FILTER:
    scope_label = "sitewide"
elif "*" in PAGE_PATH_FILTER:
    scope_label = f"pages matching {PAGE_PATH_FILTER}"
else:
    scope_label = f"page {PAGE_PATH_FILTER}"

chart_context_label = f"{scope_label} ({subdomain_scope_label})"

print(f"Project: {config.PROJECT_ID}")
print(f"GA4 dataset: {config.GA4_DATASET}")
print(f"Window: {ANALYSIS_START_DATE} to {ANALYSIS_END_DATE}")
print(f"Page-path filter: {PAGE_PATH_FILTER or '[sitewide]'}")
print(f"Destination scope: {subdomain_scope_label}")
print(f"Top domains: {TOP_N_DOMAINS}")


In [ ]:
query = load_sql_template(
    "sql/notebooks/ga4_outbound_domain_traffic.sql",
    project_id=config.PROJECT_ID,
    ga4_dataset=config.GA4_DATASET,
)

params = [
    bigquery.ScalarQueryParameter("start_date", "DATE", ANALYSIS_START_DATE),
    bigquery.ScalarQueryParameter("end_date", "DATE", ANALYSIS_END_DATE),
    bigquery.ScalarQueryParameter("page_path_filter", "STRING", PAGE_PATH_FILTER),
    bigquery.ScalarQueryParameter("include_lifeline_subdomains", "BOOL", INCLUDE_LIFELINE_SUBDOMAINS),
]

estimated_bytes = dry_run_bytes(client, query, params=params)
print(f"Estimated bytes scanned: {estimated_bytes:,}")

df = run_query(client, query, params=params)
print(f"Rows returned: {len(df):,}")
df.head()


In [ ]:
if df.empty:
    print("No outbound click rows found for this combination of dates and filters.")
else:
    domain_totals = (
        df[["destination_domain", "total_clicks"]]
        .drop_duplicates()
        .sort_values(["total_clicks", "destination_domain"], ascending=[False, True])
        .reset_index(drop=True)
    )
    top_domains = domain_totals.head(TOP_N_DOMAINS).copy()
    top_domain_list = top_domains["destination_domain"].tolist()
    top_df = df[df["destination_domain"].isin(top_domain_list)].copy()

    summary = pd.DataFrame(
        [
            ["Scope", scope_label],
            ["Destination scope", subdomain_scope_label],
            ["Distinct destination domains", f"{domain_totals['destination_domain'].nunique():,}"],
            ["Total outbound clicks", f"{int(domain_totals['total_clicks'].sum()):,}"],
            ["Top 10 domains shown", f"{min(TOP_N_DOMAINS, len(domain_totals)):,}"],
            ["Top destination domain", top_domains.iloc[0]['destination_domain'] if not top_domains.empty else "[none]"],
        ],
        columns=["metric", "value"],
    )
    summary


In [ ]:
if df.empty:
    print("No trend chart to plot because the query returned no outbound clicks.")
else:
    trend_df = (
        top_df.groupby(["report_date", "destination_domain"], as_index=False)["daily_clicks"]
        .sum()
        .sort_values(["destination_domain", "report_date"])
    )

    fig = px.line(
        trend_df,
        x="report_date",
        y="daily_clicks",
        color="destination_domain",
        template="lifeline",
        title=f"Daily Outbound Clicks to Top {min(TOP_N_DOMAINS, len(top_domain_list))} Domains from {chart_context_label} ({analysis_window_label})",
        labels={
            "report_date": "Date",
            "daily_clicks": "Outbound clicks",
            "destination_domain": "Destination domain",
        },
    )
    fig.update_layout(hovermode="x unified")
    lifeline_theme.add_lifeline_logo(fig)
    fig.show()


In [ ]:
if df.empty:
    print("No totals chart to plot because the query returned no outbound clicks.")
else:
    bar_df = top_domains.sort_values(["total_clicks", "destination_domain"], ascending=[True, False]).copy()
    bar_df["label"] = bar_df["total_clicks"].map(lambda value: f"{int(value):,}")

    fig = px.bar(
        bar_df,
        x="total_clicks",
        y="destination_domain",
        orientation="h",
        text="label",
        template="lifeline",
        title=f"Total Outbound Clicks to Top {min(TOP_N_DOMAINS, len(top_domain_list))} Domains from {chart_context_label} ({analysis_window_label})",
        labels={
            "total_clicks": "Outbound clicks",
            "destination_domain": "Destination domain",
        },
    )
    fig.update_traces(marker_color=lifeline_theme.SUPPORT_BLUE, textposition="outside", cliponaxis=False)
    fig.update_layout(
        height=max(420, 60 + 36 * len(bar_df)),
        margin=dict(l=220, r=120, t=90, b=60),
        yaxis_title="",
    )
    lifeline_theme.add_lifeline_logo(fig)
    fig.show()


## Interpretation notes

- `Outbound clicks` are GA4 `click` events from Lifeline pages where the destination domain comes from `link_domain` or, if needed, `link_url`.
- `www.lifeline.org.au` is always excluded from destination domains.
- With `INCLUDE_LIFELINE_SUBDOMAINS = False`, other `lifeline.org.au` destinations such as `give.lifeline.org.au` are excluded too.
- With `INCLUDE_LIFELINE_SUBDOMAINS = True`, those non-`www` Lifeline destinations can appear in the charts.
- `PAGE_PATH_FILTER` supports exact paths and `*` wildcards over normalized page paths, so `/get-help/support-toolkit*` will include that whole path family.
